In [2]:
!pip install textblob vaderSentiment scikit-learn tabulate

Defaulting to user installation because normal site-packages is not writeable
Looking in links: /usr/share/pip-wheels
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 624.3/624.3 kB 2.9 MB/s  0:00:00


In [7]:
from textblob import TextBlob
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
from sklearn.metrics import classification_report
from tabulate import tabulate

data = [
    ("I love this product, it's amazing!", 'positive'),
    ("This product is terrible, I hate it.", 'negative'),
    ("It's okay, not bad but not great either.", 'neutral'),
    ("Best product ever, highly recommended!", 'positive'),
    ("I'm really disappointed with the quality.", 'negative'),
    ("So-so product, nothing special about it.", 'neutral'),
    ("The customer service was excellent!", 'positive'),
    ("I wasted my money on this useless product.", 'negative'),
    ("It's not the worst, but certainly not the best.", 'neutral'),
    ("I can't live without this product, it's a lifesaver!", 'positive'),
    ("The product arrived damaged and unusable.", 'negative'),
    ("It's average, neither good nor bad.", 'neutral'),
    ("Highly disappointed with the purchase.", 'negative'),
    ("The product exceeded my expectations.", 'positive'),
    ("It's just okay, nothing extraordinary.", 'neutral'),
    ("This product is excellent, it exceeded all my expectations!", 'positive'),
    ("I regret purchasing this product, it's a waste of money.", 'negative'),
    ("It's neither good nor bad, just average.", 'neutral'),
    ("Outstanding customer service, highly recommended!", 'positive'),
    ("I'm very disappointed with the quality of this item.", 'negative'),
    ("It's not the best product, but it gets the job done.", 'neutral'),
    ("This product is a game-changer, I can't imagine life without it!", 'positive'),
    ("I received a defective product, very dissatisfied.", 'negative'),
    ("It's neither great nor terrible, just okay.", 'neutral'),
    ("Fantastic product, I would buy it again in a heartbeat!", 'positive'),
    ("Avoid this product at all costs, complete waste of money.", 'negative'),
    ("It's decent, but nothing extraordinary.", 'neutral'),
    ("Impressive quality, exceeded my expectations!", 'positive'),
    ("I'm very unhappy with this purchase, total disappointment.", 'negative'),
    ("It's neither amazing nor terrible, somewhere in between.", 'neutral')
]

table_data = [["Text", "Actual Label", "TextBlob Polarity", "TextBlob Sentiment", "VADER Compound", "VADER Sentiment"]]

for text, actual_label in data:
    blob = TextBlob(text)
    tb_polarity = blob.sentiment.polarity

    if tb_polarity > 0:
        tb_label = 'positive'
    elif tb_polarity < 0:
        tb_label = 'negative'
    else:
        tb_label = 'neutral'

    analyzer = SentimentIntensityAnalyzer()
    vs = analyzer.polarity_scores(text)
    vader_compound = vs['compound']

    if vader_compound > 0.05:
        vader_label = 'positive'
    elif vader_compound < -0.05:
        vader_label = 'negative'
    else:
        vader_label = 'neutral'

    table_data.append([text, actual_label, tb_polarity, tb_label, vader_compound, vader_label])

print(tabulate(table_data, headers="firstrow", tablefmt="plain"))

actual_labels = [row[1] for row in table_data[1:]]
tb_labels = [row[3] for row in table_data[1:]]
vader_labels = [row[5] for row in table_data[1:]]

tb_classification_report = classification_report(actual_labels, tb_labels, target_names=['negative', 'neutral', 'positive'])

vader_classification_report = classification_report(actual_labels, vader_labels, target_names=['negative', 'neutral', 'positive'])

print("\nClassification Report for TextBlob:")
print(tb_classification_report)

print("\nClassification Report for VADER:")
print(vader_classification_report)

Text                                                              Actual Label      TextBlob Polarity  TextBlob Sentiment      VADER Compound  VADER Sentiment
I love this product, it's amazing!                                positive                  0.625      positive                        0.8516  positive
This product is terrible, I hate it.                              negative                 -0.9        negative                       -0.7783  negative
It's okay, not bad but not great either.                          neutral                   0.15       positive                       -0.4707  negative
Best product ever, highly recommended!                            positive                  0.6        positive                        0.7639  positive
I'm really disappointed with the quality.                         negative                 -0.75       negative                       -0.5256  negative
So-so product, nothing special about it.                          neutral        

In [6]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import SVC
from sklearn.metrics import classification_report


texts = [text for text, label in data]
labels = [label for text, label in data]
X_train, X_test, y_train, y_test = train_test_split(texts, labels, test_size=0.4, random_state=42)

vectorizer = CountVectorizer()
X_train = vectorizer.fit_transform(X_train)
X_test = vectorizer.transform(X_test)

nb_classifier = MultinomialNB()
svm_classifier = SVC(kernel='linear')

nb_classifier.fit(X_train, y_train)
svm_classifier.fit(X_train, y_train)

nb_classification_report = classification_report(y_test, nb_classifier.predict(X_test), target_names=['negative', 'neutral', 'positive'])

svm_classification_report = classification_report(y_test, svm_classifier.predict(X_test), target_names=['negative', 'neutral', 'positive'])

print("\nClassification Report for Naive Bayes:")
print(nb_classification_report)

print("\nClassification Report for SVM:")
print(svm_classification_report)


Classification Report for Naive Bayes:
              precision    recall  f1-score   support

    negative       0.80      1.00      0.89         4
     neutral       0.75      1.00      0.86         3
    positive       1.00      0.60      0.75         5

    accuracy                           0.83        12
   macro avg       0.85      0.87      0.83        12
weighted avg       0.87      0.83      0.82        12


Classification Report for SVM:
              precision    recall  f1-score   support

    negative       0.75      0.75      0.75         4
     neutral       0.75      1.00      0.86         3
    positive       0.75      0.60      0.67         5

    accuracy                           0.75        12
   macro avg       0.75      0.78      0.76        12
weighted avg       0.75      0.75      0.74        12

